In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = "2,3,4,7"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [3]:
model_path = "/mnt/petrelfs/share_data/songmingyang/model/reasoning/policy_models/DeepSeek-R1-Distill-Qwen-7B"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path).cuda()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [14]:
prompt = "Your task is to follow a systematic, thorough reasoning process before providing the final solution. This involves analyzing, summarizing, exploring, reassessing, and refining your thought process through multiple iterations. Structure your response into two sections: Thought and Solution. In the Thought section, present your reasoning using the format: “<think>\\n {thoughts} </think>\\n”. Each thought should include detailed analysis, brainstorming, verification, and refinement of ideas. After “</think>\\n,” in the Solution section, provide the final, logical, and accurate answer, clearly derived from the exploration in the Thought section. If applicable, include the answer in \\boxed{} for closed-form results like multiple choices or mathematical solutions."
problem = "If $n$ is a natural number less than 50, find all values of $n$ such that the values of the algebraic expressions $4n+5$ and $7n+6$ have a common divisor greater than 1."
answer1 = "x\\in[4;\\infty)"
answer2 = "<think> ## Solution. The given inequality is equivalent to the system of inequalities $$ \left\{\begin{array} { l } { x ^ { 2 } - x - 12 \geq 0 , } \\ { x > 0 , } \\ { x ^ { 2 } - x - 12 > 0 } \\ { x > -12 } \end{array}\right.} \\ \text { is } \end{array}\right.\right. $$ Using the number line, we get $x \in [4 ; \infty)$. ![](https://cdn.mathpix.com/cropped/2024_05_22_fa9db84b44b98ec0a5c7g-533.jpg?height=136&width=642&top_left_y=160&top_left_x=447) </think> Answer: \\boxed{$\quad x \in [4 ; \infty)$}."
answer3 = "14. The answer is 48. $A T$ is a radius of the circle $C_{1}$; since $A D$ is also a radius, it measures 80. Furthermore, the angle $A \widehat{D} B$ is a right angle; by the Pythagorean theorem, then $B D$ measures $\sqrt{(80+90)^{2}-80^{2}}=150$. But the triangles $A B D$ and $B T C$ are similar (they are right triangles and have the angle at $B$ in common), so $B D: B T = B A: B C$ and $B C$ measures 102, therefore the length of $C D$ is 48. ![](https://cdn.mathpix.com/cropped/2024_04_17_cf63dde6cb9f14f4f823g-10.jpg?height=371&width=694&top_left_y=2170&top_left_x=1155)"
gene_answer = "**\\n\\n</think>\\n\\nTo solve the problem, we need to find all natural numbers"

In [19]:
input_token1 = tokenizer(prompt+problem+answer1, return_tensors='pt').to(model.device)
input_token2 = tokenizer(prompt+problem+answer2, return_tensors='pt').to(model.device)
input_token3 = tokenizer(prompt+problem+answer3, return_tensors='pt').to(model.device)
input_token_gene = tokenizer(prompt+problem+gene_answer, return_tensors='pt').to(model.device)
input_token = tokenizer(prompt+problem, return_tensors='pt').to(model.device)

In [20]:
output1 = model(**input_token1, use_cache=False)
output2 = model(**input_token2, use_cache=False)
output3 = model(**input_token3, use_cache=False)
output_gene = model(**input_token_gene, use_cache=False)

In [41]:
print(input_token['input_ids'].shape)

torch.Size([1, 193])


In [39]:
response_ids =input_token_gene['input_ids'][:,193:]

shift_logits = torch.softmax(output_gene.logits[:,192:-1], dim=-1)
logits = torch.softmax(output_gene.logits[:,193:], dim=-1)
score = torch.gather(logits, dim=-1, index=response_ids.unsqueeze(-1)).squeeze(-1)
print("score:\n", score)
shift_score = torch.gather(shift_logits, dim=-1, index=response_ids.unsqueeze(-1)).squeeze(-1)
print("\nshift_score:\n", shift_score)

score:
 tensor([[1.5603e-04, 2.0221e-07, 4.3708e-09, 6.3794e-03, 9.4850e-08, 2.6997e-01,
         5.8859e-04, 5.6058e-09, 6.0069e-08, 6.0412e-10, 1.8902e-07, 1.8359e-08,
         2.2063e-07, 1.9462e-09, 3.4102e-09, 3.2681e-08, 1.5573e-08, 1.4991e-09,
         1.6841e-09]], device='cuda:0', grad_fn=<SqueezeBackward1>)

shift_score:
 tensor([[0.0106, 0.0039, 0.9690, 0.2549, 0.0047, 0.0178, 0.2700, 0.3276, 0.3867,
         0.7816, 0.9997, 0.7748, 0.9588, 0.8800, 0.9994, 0.9756, 0.9855, 0.9954,
         1.0000]], device='cuda:0', grad_fn=<SqueezeBackward1>)


In [63]:
test_str = "Thank you ."
test_ids = tokenizer(test_str, return_tensors='pt').to(model.device)
output = model(**test_ids, use_cache=False)
logits = torch.softmax(output.logits, dim=-1)
print(test_ids['input_ids'].shape)
print(logits.shape)
print(torch.gather(logits[:,:-1], dim=-1, index=test_ids['input_ids'][:,1:].unsqueeze(-1)).squeeze(-1))
print(torch.gather(logits[:,1:], dim=-1, index=test_ids['input_ids'][:,1:].unsqueeze(-1)).squeeze(-1))

torch.Size([1, 4])
torch.Size([1, 4, 152064])
tensor([[7.0173e-06, 2.0315e-05, 3.1313e-04]], device='cuda:0',
       grad_fn=<SqueezeBackward1>)
tensor([[7.4305e-06, 2.1747e-05, 2.9597e-02]], device='cuda:0',
       grad_fn=<SqueezeBackward1>)


In [7]:
solution_len1 = solution_token1['input_ids'].size(1)
solution_len2 = solution_token2['input_ids'].size(1)
solution_len3 = solution_token3['input_ids'].size(1)

In [8]:
def compute_reward(logits, responses_input_ids, temperature=1):
    response_length = responses_input_ids.size(1)
    logits = torch.softmax(logits, dim=-1)/temperature
    logits = logits[:, -response_length - 1:-1]  # (bsz, response_length,vacab_size)
    token_logits = torch.gather(logits, dim=-1, index=responses_input_ids.unsqueeze(-1)).squeeze(-1)
    return token_logits

In [9]:
def compute_reward_log(logits, responses_input_ids, temperature=1.0):
    response_length = responses_input_ids.size(1)
    log_prob = torch.log_softmax(logits, dim=-1)/temperature
    log_prob = log_prob[:, -response_length - 1:-1]  # (bsz, response_length,vacab_size)
    token_reward = torch.gather(log_prob, dim=-1, index=responses_input_ids.unsqueeze(-1)).squeeze(-1)
    return token_reward

In [10]:
outputs = model.generate(**input_token, 
                             max_new_tokens=8192,
                             do_sample=True,
                             temperature=1.0,
                             pad_token_id=tokenizer.eos_token_id)
print(tokenizer.batch_decode(outputs[:, input_token['input_ids'].size(1):]))

["**\n\n</think>\n\nTo solve the problem, we need to find all natural numbers \\( n \\) less than 50 such that the algebraic expressions \\( 4n + 5 \\) and \\( 7n + 6 \\) have a common divisor greater than 1.\n\n**Step 1: Express the Problem Mathematically**\n\nWe need to find \\( n \\) such that:\n\\[\n\\gcd(4n + 5, 7n + 6) > 1\n\\]\n\n**Step 2: Use the Euclidean Algorithm**\n\nThe greatest common divisor (gcd) of two numbers can be found using the Euclidean algorithm. Applying this to \\( 4n + 5 \\) and \\( 7n + 6 \\):\n\\[\n\\gcd(4n + 5, 7n + 6) = \\gcd(4n + 5, 7n + 6 - (4n + 5)) = \\gcd(4n + 5, 3n + 1)\n\\]\n\\[\n\\gcd(4n + 5, 3n + 1) = \\gcd(3n + 1, 4n + 5 - (3n + 1)) = \\gcd(3n + 1, n + 4)\n\\]\n\\[\n\\gcd(3n + 1, n + 4) = \\gcd(n + 4, 3n + 1 - 3(n + 4)) = \\gcd(n + 4, -11) = \\gcd(n + 4, 11)\n\\]\n\n**Step 3: Determine When the GCD is Greater Than 1**\n\nThe gcd is greater than 1 if and only if 11 divides \\( n + 4 \\):\n\\[\n11 \\mid (n + 4) \\quad \\Rightarrow \\quad n + 4 = 1

In [11]:
print(compute_reward(output1.logits, solution_token1['input_ids']))
print(compute_reward(output2.logits, solution_token2['input_ids']))
print(compute_reward(output3.logits, solution_token3['input_ids']))

tensor([[6.0642e-12, 1.9667e-08, 8.2669e-05, 1.0667e-02, 1.9939e-04, 5.0454e-03,
         9.0082e-05, 6.3593e-01, 9.9976e-01, 1.7194e-01]], device='cuda:0',
       grad_fn=<SqueezeBackward1>)
tensor([[6.6402e-10, 7.6045e-03, 5.9287e-10, 2.1589e-02, 1.8435e-04, 4.1064e-02,
         1.4803e-02, 9.9349e-05, 7.4217e-01, 2.4369e-03, 9.9768e-01, 3.9953e-02,
         5.9053e-01, 2.6603e-01, 7.4865e-01, 1.1898e-03, 1.4352e-01, 1.0734e-01,
         8.6732e-01, 7.5802e-01, 3.3830e-09, 4.4075e-01, 9.9950e-01, 7.6560e-01,
         4.5373e-02, 1.6704e-02, 2.1762e-02, 9.2663e-01, 5.8345e-01, 1.3602e-01,
         5.3197e-02, 9.0552e-01, 9.4642e-01, 9.2946e-01, 9.9749e-01, 3.2603e-01,
         2.8476e-01, 4.2691e-01, 9.2705e-01, 3.9443e-01, 1.4675e-01, 4.3355e-01,
         6.0285e-01, 9.7201e-01, 8.6675e-01, 9.7833e-01, 1.9881e-03, 9.3920e-01,
         2.0038e-01, 8.6826e-01, 7.8014e-01, 5.7694e-03, 8.0911e-01, 8.9852e-01,
         1.1715e-01, 9.8233e-01, 7.1008e-01, 1.4924e-01, 8.6023e-01, 1.0153e-01

In [12]:
log_prob_mean1 = compute_reward_log(output1.logits, solution_token1['input_ids']).mean(dim=-1)
log_prob_mean2 = compute_reward_log(output2.logits, solution_token2['input_ids']).mean(dim=-1)
log_prob_mean3 = compute_reward_log(output3.logits, solution_token3['input_ids']).mean(dim=-1)

print("log_prob_mean:",log_prob_mean1.item(), log_prob_mean2.item(), log_prob_mean3.item())
print("prob_mean:",log_prob_mean1.exp().item(), log_prob_mean2.exp().item(), log_prob_mean3.exp().item())

log_prob_mean: -8.285204887390137 -2.369258165359497 -2.1012492179870605
prob_mean: 0.0002522209833841771 0.09355010092258453 0.1223035529255867


In [13]:
gene = tokenizer.batch_decode(outputs[:, input_token['input_ids'].size(1):])[0]
gene_input_token = tokenizer(prompt+problem+gene, return_tensors='pt').to(model.device)
solution_token_gene = tokenizer(gene, return_tensors='pt').to(model.device)
output_gene = model(**gene_input_token, use_cache=False)
prob=compute_reward(output_gene.logits, solution_token_gene['input_ids'])
print(prob)
log_prob = compute_reward_log(output_gene.logits, solution_token_gene['input_ids']).mean(dim=-1)
print("log_prob_mean:",log_prob.item(), "prob_mean:",log_prob.exp().item())

tensor([[6.6402e-10, 2.2663e-01, 3.9732e-02,  ..., 9.9788e-01, 9.2462e-01,
         9.9988e-01]], device='cuda:0', grad_fn=<SqueezeBackward1>)
log_prob_mean: -0.10962716490030289 prob_mean: 0.8961682319641113


In [27]:
print(torch.softmax(output1.logits[:, -solution_len1 - 1:-1], dim=-1).argmax(dim=-1))

tensor([[   16,   382,   198,   709,    59,    16,    11,   258, 36958, 15087]],
       device='cuda:0')


In [28]:
print(tokenizer.batch_decode([16,   382,   198,   709,    59,    16,    11,   258, 36958, 15087]))
print(tokenizer.batch_decode(solution_token1['input_ids']))

['1', '.\n\n', '\n', 'ge', '\\', '1', ',', 'in', 'fty', ')$']
['<｜begin▁of▁sentence｜>x\\in[4;\\infty)']


In [29]:
print(compute_reward(output3.logits, solution_token3['input_ids']))
print(compute_reward_log(output3.logits, solution_token3['input_ids']))

tensor([[6.6402e-10, 1.4122e-04, 1.3465e-02, 5.6580e-01, 2.9892e-02, 7.8658e-02,
         4.8346e-01, 9.0308e-02, 1.1158e-01, 5.8167e-02, 4.4559e-01, 1.9499e-04,
         1.4769e-03, 9.4977e-06, 1.9139e-01, 2.7163e-01, 3.7215e-01, 5.4823e-05,
         7.5347e-01, 1.9016e-01, 8.7265e-01, 2.9298e-02, 8.0556e-02, 8.7896e-02,
         7.7951e-01, 8.3123e-01, 5.5483e-03, 5.0148e-04, 6.7336e-01, 5.1096e-01,
         1.9983e-04, 9.1391e-01, 9.1151e-01, 4.9310e-02, 9.8371e-01, 9.8478e-01,
         5.1078e-01, 4.3767e-02, 1.4986e-04, 4.5984e-01, 2.9835e-02, 4.6235e-03,
         1.6068e-01, 4.0047e-04, 9.9686e-01, 9.3625e-02, 2.8241e-02, 3.0842e-01,
         5.1123e-01, 1.8375e-03, 2.4992e-02, 9.9976e-01, 8.9308e-01, 1.6647e-01,
         8.8189e-01, 6.0816e-01, 9.5637e-01, 9.3478e-01, 3.0895e-01, 9.9148e-01,
         9.9886e-01, 1.1589e-02, 5.2002e-03, 8.3523e-01, 8.8680e-01, 9.9984e-01,
         9.9642e-01, 9.8707e-01, 9.8246e-01, 1.6234e-03, 1.8483e-01, 1.9807e-01,
         9.5962e-01, 6.0685e

In [30]:
print(compute_reward(output2.logits, solution_token2['input_ids']))
print(compute_reward_log(output2.logits, solution_token2['input_ids']))

tensor([[6.6402e-10, 7.6045e-03, 5.9287e-10, 2.1589e-02, 1.8435e-04, 4.1064e-02,
         1.4803e-02, 9.9349e-05, 7.4217e-01, 2.4369e-03, 9.9768e-01, 3.9953e-02,
         5.9053e-01, 2.6603e-01, 7.4865e-01, 1.1898e-03, 1.4352e-01, 1.0734e-01,
         8.6732e-01, 7.5802e-01, 3.3830e-09, 4.4075e-01, 9.9950e-01, 7.6560e-01,
         4.5373e-02, 1.6704e-02, 2.1762e-02, 9.2663e-01, 5.8345e-01, 1.3602e-01,
         5.3197e-02, 9.0552e-01, 9.4642e-01, 9.2946e-01, 9.9749e-01, 3.2603e-01,
         2.8476e-01, 4.2691e-01, 9.2705e-01, 3.9443e-01, 1.4675e-01, 4.3355e-01,
         6.0285e-01, 9.7201e-01, 8.6675e-01, 9.7833e-01, 1.9881e-03, 9.3920e-01,
         2.0038e-01, 8.6826e-01, 7.8014e-01, 5.7694e-03, 8.0911e-01, 8.9852e-01,
         1.1715e-01, 9.8233e-01, 7.1008e-01, 1.4924e-01, 8.6023e-01, 1.0153e-01,
         9.9208e-01, 9.8613e-01, 8.6226e-01, 9.9850e-01, 5.8532e-01, 6.6354e-01,
         9.1386e-01, 9.9115e-01, 7.8207e-01, 9.5321e-01, 2.8472e-02, 9.1606e-01,
         8.7908e-01, 2.8518e

In [31]:
print(compute_reward_log(output1.logits, solution_token1['input_ids']))
print(output1.logits)

tensor([[-1.7778,  2.4098, 14.5240, 12.3873,  7.9572, 14.3737,  2.4327, 12.9555,
          8.3120, 12.1297]], device='cuda:0', grad_fn=<SqueezeBackward1>)
tensor([[[-0.2406,  1.6353,  0.8848,  ..., -2.0610, -2.0603, -2.0605],
         [ 0.8998,  4.4132,  2.3070,  ..., -2.5472, -2.5461, -2.5463],
         [12.6013,  8.0565,  9.5498,  ...,  4.5140,  4.5150,  4.5152],
         ...,
         [ 3.5675,  1.0368,  2.2169,  ..., -4.9149, -4.9143, -4.9143],
         [ 3.4778,  6.1058,  3.7671,  ..., -2.0685, -2.0675, -2.0677],
         [ 0.3190, -0.2401,  3.4817,  ...,  0.0546,  0.0559,  0.0551]]],
       device='cuda:0', grad_fn=<UnsafeViewBackward0>)
